I want to copmare the NormProd output between the original NCI TIFFs GA provided to UTAS vs the AWS pipeline using the notebook 

/g/data/yp75/sb2020/work/de-sar-sample-data/demo_notebooks/extended_examples/loading_EW_data_with_stac_geoparquet.ipynb

In [ ]:
import os
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim

In [ ]:
aws_root = "/g/data/yp75/sb2020/DATA/Prydz_AWS_test"
utas_root = "/g/data/yp75/sb2020/DATA/Prydz"

In [ ]:
def read_rgb(path):
    with rasterio.open(path) as src:
        rgb = src.read([1, 2, 3]).astype("float32")
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min())
    return rgb.transpose(1, 2, 0)

aws_folders = [
    f for f in os.listdir(aws_root)
    if f.startswith("S1_image_pair_")
]

results = []  # store metrics for summary

for folder in aws_folders:

    aws_tif = os.path.join(
        aws_root, folder, "normprod_smovar_RGB_resampled_10_10.tif"
    )
    utas_tif = os.path.join(
        utas_root, folder, "normprod_smovar_RGB_resampled_10_10.tif"
    )

    if not os.path.exists(utas_tif):
        print(f"Missing UTAS file for {folder}, skipping.")
        continue

    # Load images
    img1 = read_rgb(aws_tif)
    img2 = read_rgb(utas_tif)

    # Align sizes
    h = min(img1.shape[0], img2.shape[0])
    w = min(img1.shape[1], img2.shape[1])
    img1_c = img1[:h, :w, :]
    img2_c = img2[:h, :w, :]

    # -----------------------------
    # METRICS
    # -----------------------------
    mean_diff = np.mean(img1_c - img2_c)
    max_diff = np.max(np.abs(img1_c - img2_c))
    pct_identical = np.mean(np.all(img1_c == img2_c, axis=2)) * 100

    gray1 = img1_c.mean(axis=2)
    gray2 = img2_c.mean(axis=2)
    ssim_score, diff_map = ssim(gray1, gray2, data_range=1.0, full=True)

    # Store for summary
    results.append({
        "pair": folder,
        "mean_diff": mean_diff,
        "max_diff": max_diff,
        "pct_identical": pct_identical,
        "ssim": ssim_score
    })

    # Print per-pair summary
    print(f"""
==============================
Comparing: {folder}
==============================
Mean difference: {mean_diff:.6f} units
Max difference: {max_diff:.6f} units
Identical pixels: {pct_identical:.4f}%
SSIM score: {ssim_score:.6f}
""")

    # -----------------------------
    # VISUALISATION (2×2 grid)
    # -----------------------------

    # Enhanced absolute difference (stretch to full 0–1 range)
    abs_diff = np.abs(img1_c - img2_c).mean(axis=2)
    abs_diff_norm = (abs_diff - abs_diff.min()) / (abs_diff.max() - abs_diff.min())

    plt.figure(figsize=(12,10))

    # --- Top-left: AWS ---
    plt.subplot(2,2,1)
    plt.imshow(img1_c)
    plt.title("AWS EW Pipeline - NormProd")
    plt.axis("off")

    # --- Top-right: UTAS ---
    plt.subplot(2,2,2)
    plt.imshow(img2_c)
    plt.title("UTAS NCI EW Pipeline - NormProd")
    plt.axis("off")

    # --- Bottom-left: Abs Diff (stretched) ---
    plt.subplot(2,2,3)
    im1 = plt.imshow(abs_diff_norm, cmap="inferno")
    plt.title("Absolute Difference")
    plt.axis("off")
    plt.colorbar(im1, fraction=0.046, pad=0.04)

    # --- Bottom-right: SSIM Map ---
    plt.subplot(2,2,4)
    im2 = plt.imshow(diff_map, cmap="viridis")
    plt.title("SSIM Map")
    plt.axis("off")
    plt.colorbar(im2, fraction=0.046, pad=0.04)

    plt.suptitle(folder, fontsize=14)
    plt.tight_layout()
    plt.show()

# ============================================================
# FINAL SUMMARY ACROSS ALL PAIRS
# ============================================================

mean_mean_diff = np.mean([r["mean_diff"] for r in results])
mean_max_diff = np.mean([r["max_diff"] for r in results])
mean_identical = np.mean([r["pct_identical"] for r in results])
mean_ssim = np.mean([r["ssim"] for r in results])

print("""
========================================
Overall Summary Across All Image Pairs
========================================
""")

print(f"Average mean difference: {mean_mean_diff:.6f} units")
print(f"Average max difference: {mean_max_diff:.6f} units")
print(f"Average identical pixels: {mean_identical:.4f}%")
print(f"Average SSIM score: {mean_ssim:.6f}")

print("""
========================================
How to Interpret These Metrics
========================================

Mean difference (units):
    • Measures average pixel intensity difference (0–1 scale)
    • Values near 0 → extremely similar
    • Values > 0.01 → noticeable differences

Max difference (units):
    • Largest single-pixel difference (0–1 scale)
    • 1.0 means at least one pixel differs by the full range
    • Often caused by isolated bright pixels or noise

Identical pixels (%):
    • Percentage of pixels that match exactly in all 3 channels
    • Float images rarely match exactly due to rounding
    • High values (>50%) indicate extremely close outputs

SSIM score (0–1):
    • Measures structural similarity (texture, edges, patterns)
    • 1.0 → identical structure
    • >0.95 → extremely similar
    • 0.8–0.95 → moderate differences
    • <0.8 → significant structural changes
""")


---

I think that a big portion of differences here are coming from the edge. so if you ignore those i think these stats would be better

I tried to pad the images and mask the border but couldn't figure out how to do it properly :(

---